# 侧信道防护实验核心代码

本 Notebook 按照"分析 SGX 缺页侧信道攻击面 → 揭示原始代码的控制流泄露 → 实现数据无关（Oblivious）CRUD → 通过逻辑访问计数验证防护一致性"的顺序整理。

代码都在 `SocProject-Trusted-Password-Manager-built-on-Intel-SGX/sgx-vault-common-new/side_channel_defense` 下，原始（无防护）基线在 `sgx-vault-common-new/Enclave/Enclave.cpp`。

## 核心结论

SGX 虽然对 Enclave 内存提供硬件级加密和完整性保护，但**恶意操作系统仍控制页表**，可通过观察缺页（Page Fault）的地址和时机反推 Enclave 内部的秘密相关访问模式。原始密码库代码中 `find_service()` 在找到目标条目后**提前返回**，导致不同目标位置的扫描槽位数不同，从而在缺页轨迹上泄露目标位置信息。

本实验通过实现**数据无关（Data-Oblivious）访问**——每次操作固定扫描全部 128 个槽位、所有比较使用恒定时间位运算、用掩码而非分支提取结果——消除了访问模式与秘密数据的相关性。逻辑访问计数测试验证了：无论查询第 1 条、第 2 条还是不存在的条目，内存访问次数完全一致。

SGX 本身的不足来自 **Enclave 代码的控制流和数据访问模式对不可信 OS 可见**（通过页表），而不是 SGX 的机密性或完整性机制失效。

## 核心文件与行号总览

| 作用 | 文件 | 行号 |
|---|---|---|
| 共享常量（槽位数、字段长度） | `Include/VaultTypes.h` | 8–11 |
| 原始易受攻击的查找函数 | `Enclave/Enclave.cpp` | 81–89 |
| 原始依赖索引的读取 | `Enclave/Enclave.cpp` | 235–260 |
| 原始依赖索引的写入 | `Enclave/Enclave.cpp` | 262–280 |
| 防护版固定大小数据结构 | `side_channel_defense/ProtectedVaultCore.h` | 12–33 |
| 恒定时间比较原语 | `side_channel_defense/ProtectedVaultCore.cpp` | 6–29 |
| 全表掩码查找 | `side_channel_defense/ProtectedVaultCore.cpp` | 37–56 |
| 掩码提取读取结果 | `side_channel_defense/ProtectedVaultCore.cpp` | 118–139 |
| 全表条件写回（update） | `side_channel_defense/ProtectedVaultCore.cpp` | 141–165 |
| 全表条件写回（add） | `side_channel_defense/ProtectedVaultCore.cpp` | 190–227 |
| 全表条件写回（delete） | `side_channel_defense/ProtectedVaultCore.cpp` | 167–188 |
| 逻辑访问计数一致性测试 | `side_channel_defense/test_oblivious_core.cpp` | 18–70 |
| 防护版构建说明 | `side_channel_defense/Makefile` | 1–14 |
| 威胁模型边界与接入指南 | `side_channel_defense/README.md` | 1–64 |

## 1. 攻击面：原始代码的缺页侧信道漏洞

源文件：`Enclave/Enclave.cpp`，第 **81–89 行**（`find_service`）和第 **235–260 行**（`ecall_vault_get`）。

### 1.1 查找函数的提前返回

`find_service` 使用标准 `strcmp` 逐槽位比较，**找到匹配后立即 `return`**。恶意 OS 通过清空页表项并观察缺页地址，可统计本次 ECALL 访问了多少个槽位的数据页：

- 目标在第 2 个槽位 → 仅扫描 2 个槽位
- 目标在第 100 个槽位 → 扫描 100 个槽位
- 目标不存在 → 扫描全部 128 个槽位

```cpp
static int find_service(const char *service)
{
    for (uint32_t i = 0; i < VAULT_MAX_CREDENTIALS; ++i)
    {
        if (g_vault.credentials[i].in_use != 0 &&
            strcmp(g_vault.credentials[i].service, service) == 0)
            return static_cast<int>(i);  // ← 提前返回！缺页次数与目标位置相关
    }
    return -1;
}
```

### 1.2 读取时的索引直接访问

获取操作通过 `find_service` 返回的索引直接访问目标槽位，进一步加剧了访问模式的不对称性：

```cpp
int ecall_vault_get(const char *service, char *username, size_t username_capacity,
                    char *password, size_t password_capacity)
{
    // ... 参数校验 ...
    const int index = find_service(service);  // ← 缺页次数泄露目标位置
    if (index < 0)
        return VAULT_ERR_NOT_FOUND;
    const CredentialWire &c = g_vault.credentials[index];  // ← 只访问一个槽位
    // ...
    copy_string(username, username_capacity, c.username);   // ← 仅复制目标数据
    copy_string(password, password_capacity, c.password);
    return VAULT_OK;
}
```

> **攻击者视角**：恶意 OS 可以反复触发同一 ECALL，每次清空不同的页并观察哪些页发生缺页，逐步推断出目标记录在 128 个槽位中的大致位置。

## 2. 防护数据结构：固定大小输入与掩码选择

源文件：`side_channel_defense/ProtectedVaultCore.h`，第 **12–33 行**（固定大小输入结构），以及第 **40–44 行**（访问统计结构）。

### 2.1 固定大小输入

所有输入字段使用固定大小数组而非可变长度 C 字符串。调用前必须将整个缓冲区清零再填充，确保每次操作触碰相同字节数：

```cpp
/* 固定大小的 Enclave 内部输入，调用前必须完整清零并填充。 */
struct ProtectedServiceInput {
    char bytes[VAULT_SERVICE_MAX + 1];   // 65 字节
};

struct ProtectedUsernameInput {
    char bytes[VAULT_USERNAME_MAX + 1];  // 129 字节
};

struct ProtectedPasswordInput {
    char bytes[VAULT_PASSWORD_MAX + 1];  // 257 字节
};
```

### 2.2 访问统计结构

`ProtectedAccessStats` 用于调试验证——不是安全边界，但可以检测实现是否真正做到了数据无关：

```cpp
struct ProtectedAccessStats {
    uint64_t slots_scanned;       // 扫描的槽位数
    uint64_t service_bytes_read;  // 服务名字节读取数
    uint64_t username_bytes_read; // 用户名字节读取数
    uint64_t password_bytes_read; // 密码字节读取数
    uint64_t slots_written;       // 写入的槽位数
};
```

> 生产环境中将 `ProtectedAccessStats *` 参数传 `nullptr` 即可关闭计数。

## 3. 恒定时间比较原语

源文件：`side_channel_defense/ProtectedVaultCore.cpp`，第 **6–29 行**。

所有秘密相关比较均通过位运算实现，**不产生依赖秘密值的分支跳转**，从而防止编译器优化为条件跳转后泄露控制流：

```cpp
/* 1 当且仅当 value == 0；不使用秘密相关跳转。 */
static uint32_t ct_is_zero_u32(uint32_t value) {
    return ((value | (0u - value)) >> 31) ^ 1u;
}

static uint32_t ct_eq_u32(uint32_t left, uint32_t right) {
    return ct_is_zero_u32(left ^ right);
}

/* 根据 bit（0 或 1）生成全 0 或全 1 的字节掩码 */
static uint8_t ct_mask_u8(uint32_t bit) {
    return static_cast<uint8_t>(0u - (bit & 1u));
}

/* 无分支选择：bit==1 返回 when_true，否则返回 when_false */
static uint32_t ct_select_u32(uint32_t bit, uint32_t when_true, uint32_t when_false) {
    const uint32_t mask = 0u - (bit & 1u);
    return (when_true & mask) | (when_false & ~mask);
}
```

### 3.1 恒定时间逐字节字符串比较

替代 `strcmp`，始终比较完整的固定长度，不提前退出：

```cpp
static uint32_t ct_equal_bytes(const char *left, const char *right, size_t length,
                               ProtectedAccessStats *stats) {
    uint32_t difference = 0;
    for (size_t i = 0; i < length; ++i) {
        difference |= static_cast<uint8_t>(left[i]) ^ static_cast<uint8_t>(right[i]);
        if (stats != nullptr) ++stats->service_bytes_read;
    }
    return ct_is_zero_u32(difference);  // 逐字节累计差异，最后一次性判断
}
```

> `difference` 是所有字节 XOR 的 OR 累积值，循环结束后才判断是否为零。**无论匹配与否，循环都执行完整的 `length` 次**。

## 4. 全表掩码查找（替代 find_service）

源文件：`side_channel_defense/ProtectedVaultCore.cpp`，第 **37–56 行**。

这是防护方案的核心——**始终遍历全部 128 个槽位**，用掩码记录第一个匹配的索引，绝不提前退出：

```cpp
static uint32_t find_match_masked(const ProtectedVaultWire *vault,
                                  const ProtectedServiceInput *service,
                                  uint32_t *selected_index,
                                  ProtectedAccessStats *stats) {
    uint32_t found = 0;
    uint32_t selected = 0;

    for (uint32_t i = 0; i < VAULT_MAX_CREDENTIALS; ++i) {  // ← 始终 128 次
        const ProtectedCredentialWire &slot = vault->credentials[i];
        const uint32_t equal = ct_equal_bytes(                // ← 恒定时间比较
            slot.service, service->bytes, sizeof(slot.service), stats);
        const uint32_t match = (static_cast<uint32_t>(slot.in_use) & 1u) & equal;
        const uint32_t take = match & (found ^ 1u);           // ← 只取第一个匹配
        selected = ct_select_u32(take, i, selected);          // ← 掩码选择，不分支
        found |= match;
        if (stats != nullptr) ++stats->slots_scanned;
    }

    *selected_index = selected;
    return found & 1u;  // 返回是否找到
}
```

| 对比维度 | 原始 `find_service` | 防护 `find_match_masked` |
|----------|--------------------|--------------------------|
| 比较方式 | `strcmp`（遇 `\0` 提前结束） | `ct_equal_bytes`（固定 65 字节） |
| 循环次数 | 1 ~ 128（取决于目标位置） | 固定 128 次 |
| 提前退出 | 是（`return i`） | 否 |
| 结果返回 | 直接返回索引值 | 掩码写入 `selected_index` |

## 5. 掩码提取读取结果（protected_vault_get）

源文件：`side_channel_defense/ProtectedVaultCore.cpp`，第 **118–139 行**。

查找完成后**不能直接通过 `selected_index` 访问目标槽位**（否则仍会产生与位置相关的页访问）。必须再次遍历全部 128 个槽位，用掩码选择性地提取数据：

```cpp
int protected_vault_get(const ProtectedVaultWire *vault,
                        const ProtectedServiceInput *service,
                        ProtectedUsernameInput *username,
                        ProtectedPasswordInput *password,
                        ProtectedAccessStats *stats) {
    // ... 参数校验与输出清零 ...

    uint32_t selected = 0;
    const uint32_t found = find_match_masked(vault, service, &selected, stats);

    /*
     * 不能在查找后直接访问 credentials[selected]。再次固定扫描整个表，
     * 并用掩码提取结果，使数据页轨迹不依赖目标索引。
     */
    for (uint32_t i = 0; i < VAULT_MAX_CREDENTIALS; ++i) {  // ← 再次全部 128 个槽位
        const uint8_t mask = ct_mask_u8(found & ct_eq_u32(i, selected));
        // mask == 0xFF 当且仅当 i == selected 且 found == 1
        const ProtectedCredentialWire &slot = vault->credentials[i];
        masked_copy_out(username->bytes, slot.username, mask,
                        stats == nullptr ? nullptr : &stats->username_bytes_read);
        masked_copy_out(password->bytes, slot.password, mask,
                        stats == nullptr ? nullptr : &stats->password_bytes_read);
    }

    return ct_select_u32(found, VAULT_OK, VAULT_ERR_NOT_FOUND);  // 无分支返回
}
```

### 5.1 掩码复制（masked_copy_out）

```cpp
template <size_t N>
static void masked_copy_out(char (&destination)[N], const char (&source)[N],
                            uint8_t mask, uint64_t *counter) {
    for (size_t i = 0; i < N; ++i) {  // ← 始终执行 N 次
        const uint8_t old_value = static_cast<uint8_t>(destination[i]);
        const uint8_t new_value = static_cast<uint8_t>(source[i]);
        destination[i] = static_cast<char>(
            old_value | (new_value & mask));  // mask=0xFF 时写入，mask=0x00 时保持
        if (counter != nullptr) ++*counter;
    }
}
```

> 对于非目标槽位（mask=0），`destination` 保持清零初值不变；对于目标槽位（mask=0xFF），写入完整数据。但**所有槽位都执行了相同的循环和内存访问指令序列**。

## 6. 全表条件写回（update / add / delete）

源文件：`side_channel_defense/ProtectedVaultCore.cpp`，第 **141–227 行**。

写操作同样不能只写目标槽位。所有更新、新增、删除都对全部 128 个槽位遍历并执行条件写回：

### 6.1 Update 操作（第 141–165 行）

```cpp
int protected_vault_update(ProtectedVaultWire *vault,
                           const ProtectedServiceInput *service,
                           const ProtectedUsernameInput *username,
                           const ProtectedPasswordInput *password,
                           ProtectedAccessStats *stats) {
    // ... 参数校验 ...
    uint32_t selected = 0;
    const uint32_t found = find_match_masked(vault, service, &selected, stats);
    const uint32_t version_ok = static_cast<uint32_t>(vault->state_version != UINT64_MAX);
    const uint32_t apply = found & version_ok;

    for (uint32_t i = 0; i < VAULT_MAX_CREDENTIALS; ++i) {  // ← 遍历全部 128 个槽位
        const uint8_t mask = ct_mask_u8(apply & ct_eq_u32(i, selected));
        ProtectedCredentialWire &slot = vault->credentials[i];
        masked_assign(slot.username, username->bytes, mask, ...);  // mask 决定是否真正写入
        masked_assign(slot.password, password->bytes, mask, ...);
        if (stats != nullptr) ++stats->slots_written;  // 每个槽位都计入
    }
    vault->state_version += apply;  // 无分支版本递增

    const uint32_t result = ct_select_u32(found, VAULT_OK, VAULT_ERR_NOT_FOUND);
    return ct_select_u32(version_ok, result, VAULT_ERR_INTERNAL);
}
```

### 6.2 Add 操作（第 190–227 行）

新增操作包含两次全遍历：先查找重复项，再查找空槽位，最后条件写入：

```cpp
int protected_vault_add(ProtectedVaultWire *vault,
                        const ProtectedServiceInput *service,
                        const ProtectedUsernameInput *username,
                        const ProtectedPasswordInput *password,
                        ProtectedAccessStats *stats) {
    // 第一次全遍历：查找是否已存在同名服务
    uint32_t duplicate_index = 0;
    const uint32_t duplicate = find_match_masked(vault, service, &duplicate_index, stats);

    // 第二次全遍历：查找第一个空槽位
    uint32_t empty_found = 0;
    uint32_t selected = 0;
    for (uint32_t i = 0; i < VAULT_MAX_CREDENTIALS; ++i) {
        const uint32_t empty = ct_is_zero_u32(vault->credentials[i].in_use);
        const uint32_t take = empty & (empty_found ^ 1u);
        selected = ct_select_u32(take, i, selected);
        empty_found |= empty;
        if (stats != nullptr) ++stats->slots_scanned;
    }

    // 第三次全遍历：条件写入全部槽位
    const uint32_t apply = (duplicate ^ 1u) & empty_found & version_ok;
    for (uint32_t i = 0; i < VAULT_MAX_CREDENTIALS; ++i) {
        const uint8_t mask = ct_mask_u8(apply & ct_eq_u32(i, selected));
        // 对每个槽位：写入 in_use、service、username、password（由 mask 控制是否生效）
        slot.in_use = ct_select_u8(mask, 1, slot.in_use);
        masked_assign(slot.service, service->bytes, mask, ...);
        masked_assign(slot.username, username->bytes, mask, ...);
        masked_assign(slot.password, password->bytes, mask, ...);
    }
    // ... 计数器和版本递增同样用掩码实现 ...
}
```

### 6.3 Delete 操作（第 167–188 行）

删除同样遍历全部槽位，对目标槽位以掩码方式覆盖为零：

```cpp
    const ProtectedCredentialWire zero_slot = {};
    for (uint32_t i = 0; i < VAULT_MAX_CREDENTIALS; ++i) {
        ProtectedCredentialWire &slot = vault->credentials[i];
        const uint8_t mask = ct_mask_u8(apply & ct_eq_u32(i, selected));
        uint8_t *destination = reinterpret_cast<uint8_t *>(&slot);
        const uint8_t *zeros = reinterpret_cast<const uint8_t *>(&zero_slot);
        for (size_t j = 0; j < sizeof(slot); ++j)
            destination[j] = ct_select_u8(mask, zeros[j], destination[j]);
    }
```

## 7. 逻辑访问计数一致性测试

源文件：`side_channel_defense/test_oblivious_core.cpp`，第 **18–70 行**。

测试通过比较不同查询的 `ProtectedAccessStats` 来验证数据无关性。**如果三条查询（第 1 条、第 2 条、不存在）的统计值完全一致，说明实现没有在访问次数上泄露目标信息**：

```cpp
int main() {
    ProtectedVaultWire vault = {};
    vault.format_version = VAULT_FORMAT_VERSION;
    vault.state_version = 1;

    // 插入两条记录：github (槽位 0) 和 bank (槽位 1)
    encode("github", "alice", "github-secret", &service, &username, &password);
    assert(protected_vault_add(&vault, &service, &username, &password, &stats) == VAULT_OK);
    encode("bank", "alice-bank", "bank-secret", &service, &username, &password);
    assert(protected_vault_add(&vault, &service, &username, &password, &stats) == VAULT_OK);

    ProtectedAccessStats github_stats = {};
    ProtectedAccessStats bank_stats = {};
    ProtectedAccessStats missing_stats = {};

    // 查询第 1 条记录
    assert(protected_encode_service("github", &service));
    assert(protected_vault_get(
        &vault, &service, &output_username, &output_password, &github_stats) == VAULT_OK);

    // 查询第 2 条记录
    assert(protected_encode_service("bank", &service));
    assert(protected_vault_get(
        &vault, &service, &output_username, &output_password, &bank_stats) == VAULT_OK);

    // 查询不存在的记录
    assert(protected_encode_service("missing", &service));
    assert(protected_vault_get(
        &vault, &service, &output_username, &output_password, &missing_stats)
        == VAULT_ERR_NOT_FOUND);

    // ★ 核心断言：三者逻辑访问计数必须完全一致
    assert(same_stats(github_stats, bank_stats));
    assert(same_stats(github_stats, missing_stats));

    puts("[PASS] get(first), get(second), and get(missing) have identical logical access counts");
    return 0;
}
```

> **注意**：此测试只能发现实现中的明显错误（例如某条路径提前退出了循环）。最终确定安全性仍需在真实 SGX HW 上运行缺页攻击工具，比较实际的页面访问轨迹。

## 8. 构建与接入 Enclave

源文件：`side_channel_defense/Makefile`（第 **1–14 行**）和 `side_channel_defense/README.md`（第 **28–43 行**）。

### 8.1 独立测试构建（不依赖 SGX SDK）

```makefile
CXX ?= g++
CXXFLAGS ?= -O2 -Wall -Wextra -Wpedantic -std=c++17

test_oblivious_core: ProtectedVaultCore.cpp ProtectedVaultCore.h test_oblivious_core.cpp
	$(CXX) $(CXXFLAGS) ProtectedVaultCore.cpp test_oblivious_core.cpp -o $@

test: test_oblivious_core
	./test_oblivious_core
```

### 8.2 接入 Enclave 的步骤

`ProtectedVaultWire` 的字段顺序和大小与原始 `VaultWire` 一致，接入步骤为：

1. 在 `Enclave/Enclave.cpp` 中包含 `ProtectedVaultCore.h`
2. 将原来的 `VaultWire` 换成 `ProtectedVaultWire`
3. ECALL 收到字符串后，先编码到完整清零的固定大小输入结构（`protected_encode_service` 等）
4. 用四个 `protected_vault_*` 函数替换原 CRUD 主体
5. 将 `ProtectedAccessStats` 传 `nullptr`（生产路径不输出调试计数）
6. 使用 `SGX_MODE=HW` 重新运行缺页轨迹实验验证

## 9. 威胁模型边界

源文件：`side_channel_defense/README.md`，第 **45–64 行**。

当前防护代码**没有隐藏**以下信道，论文中应明确说明这些边界：

| 未隐藏信息 | 说明 |
|-----------|------|
| 调用了哪一种 ECALL | add/get/update/delete 的类型仍可区分 |
| ECALL 的调用时刻和次数 | 时序侧信道不在防护范围内 |
| 返回状态码 | 成功/失败/未找到 等仍可区分 |
| `list` 主动输出的服务列表 | list 操作未做 oblivious 处理 |
| `seal/unseal` 的格式校验控制流 | 密封/解封流程不在防护范围内 |
| EDL `[string]` 参数长度 | 字符串封送可能暴露输入长度 |

> 如果实验要求连字符串长度也隐藏，应将 EDL 参数改为固定大小数组，如 `[in, size=VAULT_SERVICE_MAX + 1]`，App 端每次传完整清零的缓冲区。

### 编译器注意事项

无分支 C++ 写法仍可能被编译器优化改变。用于论文或最终报告时，应：

1. 对 Enclave 优化后的反汇编进行检查，确认比较和掩码选择没有被重新编译成依赖秘密值的提前跳转
2. 用真实受控缺页攻击验证轨迹，不能只根据源代码判断安全

## 10. 攻击与防护流程对比

```mermaid
flowchart TB
    subgraph 原始代码["原始代码：易受缺页侧信道攻击"]
        direction TB
        A1["ecall_vault_get('bank')"] --> A2["find_service: i=0 比较 github"]
        A2 --> A3["i=1 比较 bank → 匹配！"]
        A3 --> A4["return 1; ← 提前退出"]
        A4 --> A5["直接访问 credentials[1]"]
        A5 --> A6["复制 password 后返回"]
        A6 -.-> A7["⚡ OS 观察到：仅扫描 2 个槽位<br/>→ 推断目标在靠前位置"]
    end

    subgraph 防护代码["防护代码：数据无关 Oblivious 访问"]
        direction TB
        B1["protected_vault_get('bank')"] --> B2["find_match_masked:<br/>i=0..127 恒定时间比较全部槽位"]
        B2 --> B3["掩码记录 selected=1, found=1"]
        B3 --> B4["第二轮全遍历 i=0..127:<br/>mask=0xFF 仅当 i==1<br/>每个槽位都执行 masked_copy_out"]
        B4 --> B5["ct_select_u32 无分支返回"]
        B5 -.-> B6["🔒 OS 观察到：固定扫描 128 个槽位<br/>→ 无法推断目标位置"]
    end

    style 原始代码 fill:#ffe0e0,stroke:#cc0000
    style 防护代码 fill:#e0ffe0,stroke:#00aa00
```